# Topic: SQL: Monthly Active Users (MAU) & MoM Growth/Retention

## Definition (30-second explanation)
* MAU measures the total number of unique users who interacted with a product or performed a specific action within a given calendar month.
* It is a foundational product analytics KPI used to track overarching product growth, evaluate feature success, and model revenue.
* It is calculated by grouping data by a truncated month and taking a distinct count of user identifiers.

## Why Interviewers Ask This
* **Technical Signal:** Tests fundamental SQL aggregations, date manipulation (`DATE_TRUNC`, `EXTRACT`), and deduplication (`DISTINCT`).
* **Advanced SQL:** Often used as a stepping stone to test window functions (e.g., using `LAG()` for Month-over-Month growth).
* **Product Sense:** Assesses if you understand the business context—knowing to ask how "active" is defined (e.g., simple login vs. making a purchase).

## Core Concepts
* **Date Truncation:** Standardizing granular timestamps into monthly buckets (e.g., mapping `2024-03-15 14:02:00` to `2024-03-01`).
* **Deduplication:** Using `COUNT(DISTINCT user_id)` ensures a user with 50 events in a month is only counted once, measuring *reach*, not *volume*.
* **Windowing for MoM:** Leveraging `LAG()` over a time-ordered window to compare current aggregations against previous periods.

## When to Use
* Tracking overall product growth and long-term user engagement trends.
* Calculating derived metrics like Month-over-Month (MoM) user growth or trial-to-paid MAU ratios.
* Building executive dashboards where daily fluctuations (DAU) are too noisy.

## Limitations
* Fails to capture daily volatility or granular session behavior (use DAU or Sessionization for this).
* Can present a false sense of growth if churn is high but offset by massive (and expensive) new user acquisition.
* Definitionally ambiguous: MAU numbers are meaningless if the "active" event is too passive (e.g., an automated push notification).

## Common Comparisons
* **MAU vs. DAU:** MAU shows broad reach; DAU shows daily habit. The DAU/MAU ratio (Stickiness) measures how often monthly users engage daily.
* **DATE_TRUNC vs. EXTRACT:** `DATE_TRUNC` keeps the year/month structure intact (`2024-01-01`), whereas `EXTRACT(MONTH)` isolates the month number (`1`), which will incorrectly group January 2024 and January 2025 together if not also grouped by year.

## Common Interview Traps
* **Missing DISTINCT:** Writing `COUNT(user_id)` instead of `COUNT(DISTINCT user_id)`, which double-counts repeat visitors.
* **Forgetting the Year:** Using `EXTRACT(MONTH)` without `EXTRACT(YEAR)` on multi-year datasets.
* **Blind Grouping:** Using `GROUP BY 1` without ensuring the date column is actually the first selected column, causing silent grouping errors.
* **Timezone Ignorance:** Failing to clarify how the system handles UTC vs. local timezones at the boundary of a month.

## SQL Syntax
```sql
-- Core MAU Pattern
SELECT 
    DATE_TRUNC('month', event_date) AS activity_month,
    COUNT(DISTINCT user_id) AS monthly_active_users
FROM user_events
GROUP BY 1
ORDER BY 1;

-- MoM Growth Pattern using CTE and LAG()
WITH monthly_mau AS (
    SELECT 
        DATE_TRUNC('month', event_date) AS activity_month,
        COUNT(DISTINCT user_id) AS mau
    FROM user_events
    GROUP BY 1
)
SELECT 
    activity_month,
    mau,
    LAG(mau) OVER (ORDER BY activity_month) AS prev_month_mau,
    ROUND(100.0 * (mau - LAG(mau) OVER (ORDER BY activity_month)) / 
          LAG(mau) OVER (ORDER BY activity_month), 2) AS mom_growth_pct
FROM monthly_mau;
```

## Important Formula
* **MoM Growth Rate:** `(Current Month MAU - Previous Month MAU) / Previous Month MAU`
* **Stickiness Ratio:** `DAU / MAU` (Values closer to 1 indicate a highly habitual product).

## 45-Second Interview Answer
"To calculate Monthly Active Users, I truncate the event timestamp to the month level using `DATE_TRUNC` and group by it. Inside the aggregation, I use `COUNT(DISTINCT user_id)` to guarantee each user is only counted once per month regardless of their activity volume. If asked for Month-over-Month growth, I wrap the MAU query in a CTE and apply the `LAG()` window function, ordering by month, to retrieve the previous month's MAU for the percentage calculation. Before writing the query, I would always verify with you how we are defining an 'active' event and what timezone boundaries we should respect."


## Example Questions

**Question:** Find the top 3 months with highest MAU in 2024.

**Ideal Interview Answer:**
```sql
SELECT 
    DATE_TRUNC('month', event_date) AS activity_month,
    COUNT(DISTINCT user_id) AS mau
FROM user_events
WHERE EXTRACT(YEAR FROM event_date) = 2024
GROUP BY 1
ORDER BY mau DESC
LIMIT 3;
```

**Common Mistakes:**
* Filtering on year using `LIKE '2024%'` which is inefficient for date types.
* Forgetting `ORDER BY mau DESC` and just applying `LIMIT 3`, which returns 3 random or chronological months.

**Likely Interviewer Follow-up:**
"How would you modify this to find the top 3 months per country instead of globally?"

---

**Question:** Calculate the 3-month rolling average of MAU.

**Ideal Interview Answer:**
```sql
WITH monthly_mau AS (
    SELECT 
        DATE_TRUNC('month', event_date) AS activity_month,
        COUNT(DISTINCT user_id) AS mau
    FROM user_events
    GROUP BY 1
)
SELECT 
    activity_month,
    mau,
    AVG(mau) OVER (
        ORDER BY activity_month 
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS rolling_3m_avg_mau
FROM monthly_mau;
```

**Common Mistakes:**
* Attempting to do the rolling average in the same query block as the `COUNT(DISTINCT)`, which violates SQL order of operations.
* Using `ROWS BETWEEN 3 PRECEDING` instead of `2 PRECEDING` (current row counts as one of the three months).

**Likely Interviewer Follow-up:**
"What happens to the rolling average for the first two months in the dataset? How might you handle that edge case?"

---

**Question:** Find months where MAU dropped by more than 10% compared to previous month.

**Ideal Interview Answer:**
```sql
WITH monthly_mau AS (
    SELECT 
        DATE_TRUNC('month', event_date) AS activity_month,
        COUNT(DISTINCT user_id) AS mau
    FROM user_events
    GROUP BY 1
),
mom_growth AS (
    SELECT 
        activity_month,
        mau,
        LAG(mau) OVER (ORDER BY activity_month) AS prev_mau
    FROM monthly_mau
)
SELECT 
    activity_month,
    mau,
    prev_mau,
    (mau - prev_mau) * 100.0 / prev_mau AS growth_pct
FROM mom_growth
WHERE (mau - prev_mau) * 100.0 / prev_mau < -10.0;
```

**Common Mistakes:**
* Calculating the percentage drop incorrectly (e.g., dividing by current MAU instead of previous MAU).
* Integer division truncation: forgetting to multiply by `100.0` or cast to a decimal before dividing.

**Likely Interviewer Follow-up:**
"If you see a sudden 15% drop in MAU for a specific month, how would you go about investigating the root cause?"

---

**Question:** Calculate MAU separately for each country.

**Ideal Interview Answer:**
```sql
SELECT 
    DATE_TRUNC('month', event_date) AS activity_month,
    country,
    COUNT(DISTINCT user_id) AS mau
FROM user_events
GROUP BY 1, 2
ORDER BY 1, 2;
```

**Common Mistakes:**
* Forgetting to add `country` to the `GROUP BY` clause, causing a syntax error in strict SQL modes.
* Not ordering the results, making the output difficult for a business user to read.

**Likely Interviewer Follow-up:**
"How would you pivot this data so that the months are rows and the top 3 countries are columns?"

---

**Question:** Find users who were active in every single month of 2024 (consistently active users).

**Ideal Interview Answer:**
```sql
SELECT 
    user_id
FROM user_events
WHERE EXTRACT(YEAR FROM event_date) = 2024
GROUP BY user_id
HAVING COUNT(DISTINCT EXTRACT(MONTH FROM event_date)) = 12;
```

**Common Mistakes:**
* Using `COUNT(event_date) = 12`, which would return users who just did 12 actions in a single day.
* Overcomplicating the query with 12 self-joins or complex CTEs instead of a simple `HAVING COUNT(DISTINCT...)`.

**Likely Interviewer Follow-up:**
"What if the year hasn't finished yet, and we just want users who were active in all available months in the dataset?"

## Practice Questions:

### Q1:

**Question:** Using the classicmodels database (Assume you are using the orders table with columns: orderNumber (INT), orderDate (DATE), customerNumber (INT)), write a SQL query to calculate the Month-over-Month (MoM) Customer Retention Rate for the year 2003.

Specifically, for each month in 2003 (starting from February), calculate the percentage of customers who made a purchase in the previous month who also made a purchase in the current month.

**Answer:**
```sql
WITH monthly_activity AS (
    -- Get distinct customer-month pairs, standardized to the 1st of the month
    SELECT DISTINCT 
        customerNumber,
        DATE_FORMAT(orderDate, '%Y-%m-01') AS orderMonth
    FROM orders
),
timeline AS (
    -- Use LAG to find the previous active month for each customer
    SELECT 
        customerNumber,
        orderMonth,
        LAG(orderMonth) OVER(PARTITION BY customerNumber ORDER BY orderMonth) AS previousOrderMonth
    FROM monthly_activity
)
SELECT 
    orderMonth,
    COUNT(customerNumber) AS totalCustomers,
    SUM(CASE WHEN DATE_SUB(orderMonth, INTERVAL 1 MONTH) = previousOrderMonth THEN 1 ELSE 0 END) AS retainedCustomers,
    ROUND(100.0 * SUM(CASE WHEN DATE_SUB(orderMonth, INTERVAL 1 MONTH) = previousOrderMonth THEN 1 ELSE 0 END) / COUNT(customerNumber), 2) AS retainedPCT
FROM timeline
GROUP BY orderMonth
ORDER BY orderMonth;
```

**Common Mistakes:**
* Not standardizing the dates to the first of the month, resulting in granular daily retention instead of monthly.
* Forgetting `PARTITION BY customerNumber` in the window function, which would mistakenly compare Customer A's date with Customer B's date.
* Using an inner join instead of a left join (if using the self-join method), which entirely drops the baseline users who churned, ruining the denominator.

**Likely Interviewer Follow-up:**
"How would this query change if we wanted to calculate 'Resurrected Users' (users who were active this month, were NOT active last month, but HAVE been active at some point in the past)?"

### Q2: 
Database Schema: classicmodels

Table Name: orders

Columns: orderNumber (INT), orderDate (DATE), customerNumber (INT)

**Question 2: Using MySQL, write a query to calculate the number of New Customers acquired in each month of 2003. A customer is considered 'New' in the specific month they make their very first purchase. If they make subsequent purchases in later months, they should not be counted as new in those later months.**


**Answer:**
```sql
WITH first_orders AS (
    -- Find the absolute first purchase date for every customer
    SELECT 
        customerNumber, 
        MIN(orderDate) AS firstOrderDate
    FROM orders
    GROUP BY customerNumber
)
SELECT
    DATE_FORMAT(firstOrderDate, '%Y-%m-01') AS firstOrderMonth,
    COUNT(DISTINCT customerNumber) AS newCustomers
FROM first_orders
-- Filter the cohorts down to the specific year requested
WHERE YEAR(firstOrderDate) = 2003
GROUP BY 1
ORDER BY 1;
```

**Common Mistakes:**
* Filtering the date *inside* the CTE. (e.g., `WHERE YEAR(orderDate) = 2003`). This is a fatal error. It finds the first order *in 2003*, not the customer's actual first order across the lifetime of the database. 
* Using complex window functions (like `ROW_NUMBER()`) when a simple `MIN()` aggregation does the exact same job much more efficiently.

**Likely Interviewer Follow-up:**
"How would you join this result back to our general MAU query so we could see Total MAU alongside New Users in the same table?"